In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [2]:
!pip install -U bitsandbytes>=0.46.1 -q
print ('done')

done


In [3]:
import bitsandbytes as bnb
print(bnb.__version__)

0.50.0


In [4]:
"""
Smart MCQ Solver - Llama-3.1-Storm-8B only, zero-shot
Kaggle competition, evaluated on MAP@3.

Started this with Qwen2.5-14B-Instruct but even in 4-bit it kept pinning
one T4 right up against its 14.56GiB ceiling once activations and the
(fairly large) embedding table were accounted for. Storm-8B is ~half the
weight count of the 14B and behaves a lot better on a single T4, so
switching to it instead of fighting the memory further.

No fine-tuning - the model is prompted with the question + five options
and we read off the logit at the answer-letter position for a confidence
ranking over A-E. Top 3 by confidence go into the submission.
"""

import gc
import os

# must be set before torch touches the GPU, helps with the fragmentation
# that was causing the earlier OOM
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SUPPORTS_BF16 = torch.cuda.is_bf16_supported() if DEVICE == "cuda" else False
print(f"device: {DEVICE}, bf16 supported: {SUPPORTS_BF16}")

TRAIN_PATH = "/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv"
TEST_PATH = "/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv"
OUT_PATH = "submission.csv"

train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)
OPTION_COLS = ["A", "B", "C", "D", "E"]

# small held-out slice just to sanity check MAP@3 before burning time on
# the full test set - not used for any training since this is zero-shot
val_df = train_df.sample(frac=0.1, random_state=SEED).reset_index(drop=True)


# ---------------------------------------------------------------------------
# MAP@3
# ---------------------------------------------------------------------------
def map_at_3(y_true, ranked_preds):
    scores = []
    for true_label, preds in zip(y_true, ranked_preds):
        s = 0.0
        for rank, p in enumerate(preds[:3]):
            if p == true_label:
                s = 1.0 / (rank + 1)
                break
        scores.append(s)
    return float(np.mean(scores))


def probs_to_ranked_letters(prob_matrix):
    order = np.argsort(-prob_matrix, axis=1)[:, :3]
    return [[OPTION_COLS[i] for i in row] for row in order]


# ---------------------------------------------------------------------------
# load model, 4-bit since 14B won't fit clean on a T4/P100 otherwise
# ---------------------------------------------------------------------------
print("loading Llama-3.1-Storm-8B...")

# 8B fits comfortably in bf16/fp16 on a single T4 (~16GB), so skip 4-bit
# entirely here - one less moving part, and it sidesteps the bnb dequant
# overhead that was eating into the 14B's memory budget
model_dtype = torch.bfloat16 if SUPPORTS_BF16 else torch.float16

tokenizer = AutoTokenizer.from_pretrained("akjindal53244/Llama-3.1-Storm-8B")

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    for i in range(torch.cuda.device_count()):
        print(f"  gpu {i} free/total (GiB): "
              f"{torch.cuda.mem_get_info(i)[0] / 1e9:.2f} / {torch.cuda.mem_get_info(i)[1] / 1e9:.2f}")

model = AutoModelForCausalLM.from_pretrained(
    "akjindal53244/Llama-3.1-Storm-8B",
    torch_dtype=model_dtype,
    device_map="auto",
    low_cpu_mem_usage=True,
)
model.eval()

# grab the token id for each letter once - assumes A-E encode to a single
# token under this tokenizer, worth double checking if you swap models later
LETTER_IDS = {c: tokenizer.encode(c, add_special_tokens=False)[0] for c in OPTION_COLS}
print("letter token ids:", LETTER_IDS)


def build_prompt(row):
    opts = "\n".join(f"{c}. {row[c]}" for c in OPTION_COLS)
    return (
        "Answer the following multiple choice question. "
        "Reply with only the letter of the correct option.\n\n"
        f"Question: {row['prompt']}\n{opts}\nAnswer:"
    )


@torch.no_grad()
def score_row(row):
    messages = [{"role": "user", "content": build_prompt(row)}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    logits = model(**inputs).logits[0, -1, :]
    letter_logits = torch.tensor([logits[LETTER_IDS[c]].item() for c in OPTION_COLS])
    return F.softmax(letter_logits, dim=0).numpy()


def score_dataframe(df, log_every=25):
    out = []
    for i, (_, r) in enumerate(df.iterrows()):
        out.append(score_row(r))
        if (i + 1) % log_every == 0:
            print(f"  scored {i+1}/{len(df)}")
    return np.stack(out, axis=0)


# ---------------------------------------------------------------------------
# quick validation pass, just to see where we stand
# ---------------------------------------------------------------------------
print("\nscoring validation slice...")
val_probs = score_dataframe(val_df)
val_preds = probs_to_ranked_letters(val_probs)
val_map3 = map_at_3(val_df["answer"].tolist(), val_preds)
print(f"validation MAP@3: {val_map3:.4f}")

del val_probs
gc.collect()
torch.cuda.empty_cache()


# ---------------------------------------------------------------------------
# full test set + submission
# ---------------------------------------------------------------------------
print("\nscoring test set...")
test_probs = score_dataframe(test_df)
test_preds = probs_to_ranked_letters(test_probs)

submission = pd.DataFrame({
    "id": test_df["id"],
    "prediction": [" ".join(p) for p in test_preds],
})
submission.to_csv(OUT_PATH, index=False)
print(f"\nwrote {OUT_PATH}, {len(submission)} rows")
print(submission.head())

device: cuda, bf16 supported: True
loading Llama-3.1-Storm-8B...


config.json:   0%|          | 0.00/910 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/185 [00:00<?, ?B/s]

letter token ids: {'A': 32, 'B': 33, 'C': 34, 'D': 35, 'E': 36}

scoring validation slice...
  scored 25/200
  scored 50/200
  scored 75/200
  scored 100/200
  scored 125/200
  scored 150/200
  scored 175/200
  scored 200/200
validation MAP@3: 0.8242

scoring test set...
  scored 25/500
  scored 50/500
  scored 75/500
  scored 100/500
  scored 125/500
  scored 150/500
  scored 175/500
  scored 200/500
  scored 225/500
  scored 250/500
  scored 275/500
  scored 300/500
  scored 325/500
  scored 350/500
  scored 375/500
  scored 400/500
  scored 425/500
  scored 450/500
  scored 475/500
  scored 500/500

wrote submission.csv, 500 rows
   id prediction
0   1      E D B
1   2      A C D
2   3      C D B
3   4      A E B
4   5      C A B
